# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Source: `docs/flyrank-seo-research-march-2026.pdf`, *The State of AI-Driven SEO*, March 2026.

Both questions below are ones I would want asked of **my** work, and in both cases the paper discloses part of the issue itself. It states its evidence standard up front: ML pages are "exploratory appendix material" that "do not override direct portfolio evidence", so these are questions about how a reader might over-read it, not claims that the authors overreached.

---

### Finding 1: "What Predicts Growth?" (ML appendix, p.29)

> *Logistic regression (71% holdout accuracy) describing which sampled features separate growing from declining pages. Content age is the strongest negative signal.*

**My methodology question: what is the base rate, and was the holdout grouped by brand?**

*On the number itself.* The 71% holdout accuracy is reported without the naive floor beside it. From Finding #1 (p.6), the cohorts are 74.8K growing versus 45.6K declining, so always guessing "growing" scores about **62%**. That reframes 71% as roughly **nine points of skill rather than seventy-one**: a real effect, but a much smaller one than the bare number suggests. This is the single change I would most want made to my own work, and it is why every table I produce now carries its base rate in an adjacent row.

*On the design.* The study spans 57 brands, and pages within a brand share templates, publishing cadence, and niche. If the holdout was drawn at random over rows, some brands sit in both training and test, and the model can score well by recognising the brand rather than by learning what a declining page looks like. I am not asserting that happened. The paper does not state its split method, which is precisely why it is worth asking. **Section 2 measures this effect on my own model**, where it was worth +0.157 precision@50 of pure illusion.

*On the label.* Trend Direction is defined (p.5) as the 30-day versus previous-30-day impression change. If the "Impressions" feature is the 90-day total, it **contains** both of those windows, and part of the answer travels into the features. Section 3 tests exactly that pair on my own data.

---

### Finding 2: "The Freshness Multiplier" (Finding #4, p.9)

> *365+ day content that was refreshed within 30 days shows 3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039).*

**My methodology question: who chose which pages got refreshed, and is the outcome partly made of the input?**

*On selection.* Refreshing is not randomly assigned; an editor decides. Editors refresh pages they believe are worth saving, which usually means pages with existing demand or strategic importance. So the refreshed and un-refreshed groups likely differed **before** any refresh happened. A 57x impression gap is large enough to suggest the groups were already different, rather than that refreshing produced a 57x lift. The supportable version is *refreshed pages are associated with higher impressions in this portfolio*; the causal reading would need pages assigned to refresh independently of how promising they looked.

*On the two numbers.* Health Score is defined on p.5 as impressions (30 pts) + position (30) + CTR (20) + scroll depth (20). Impressions are therefore about **30% of the health score by construction**. Side by side, "3.2x health" and "57x impressions" read as two independent confirmations, when much of the first restates the second.

**Credit where it is due, because this is the standard I am aiming at.** The paper does this well elsewhere: it caps its own freshness finding, flagging that the `361+` bucket spikes to 283:1 "only because the sample is tiny and there is just 1 declining page"; it narrows its age claim to "not evidence that age naturally reverses performance decline on its own"; and on p.27 it states outright that health-score feature importances are "descriptive rather than causal" because the target is built from the inputs. My two questions are that same caveat, asked in two more places.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# BEFORE / AFTER: the same Week-5 random forest, scored two ways.
#   BEFORE = random ROW split  -> a client's pages land in both train and test
#   AFTER  = grouped by client -> test clients were never seen in training
import os, json
from pathlib import Path
import numpy as np, pandas as pd, sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

SEED, N_SPLITS = 42, 8
if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])

NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days",
       "days_since_last_update", "ctr", "avg_position", "engagement_rate",
       "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent", "age_tier",
       "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]

def model(num_cols):
    return Pipeline([("prep", ColumnTransformer([
        ("n", Pipeline([("i", SimpleImputer(strategy="median")),
                        ("s", StandardScaler())]), num_cols),
        ("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                        ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), CAT)])),
        ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                     class_weight="balanced", n_jobs=-1, random_state=SEED))])

def precision_at_k(scores, labels, k=50):
    return float(np.asarray(labels)[np.argsort(-np.asarray(scores))[:k]].mean())

y = df["is_declining_label"].values
before, after = [], []
for seed in range(N_SPLITS):
    r_tr, r_te = next(ShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(df))
    g_tr, g_te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
                      .split(df, y, groups=df["client_id"]))
    for tr, te, store in [(r_tr, r_te, before), (g_tr, g_te, after)]:
        prob = model(NUM).fit(df.iloc[tr][NUM + CAT], y[tr]).predict_proba(df.iloc[te][NUM + CAT])[:, 1]
        store.append(dict(p50=precision_at_k(prob, y[te]), auc=roc_auc_score(y[te], prob),
                          base=y[te].mean(),
                          shared_clients=len(set(df.iloc[tr].client_id) & set(df.iloc[te].client_id))))
B, A = pd.DataFrame(before), pd.DataFrame(after)

print("=" * 76)
print("BEFORE / AFTER  -- identical model and features, only the split changes")
print("=" * 76)
comp = pd.DataFrame({
    "BEFORE: random row split": [B.p50.mean(), B.p50.std(), B.auc.mean(), B.base.mean(),
                                 B.shared_clients.mean()],
    "AFTER: grouped by client": [A.p50.mean(), A.p50.std(), A.auc.mean(), A.base.mean(),
                                 A.shared_clients.mean()],
}, index=["P@50 mean", "P@50 sd", "ROC-AUC", "test base rate",
          "clients in BOTH train & test"]).round(3)
print(comp.to_string())

print(f"\nINFLATION bought by the dishonest split:")
print(f"  precision@50 : {B.p50.mean() - A.p50.mean():+.3f}")
print(f"  ROC-AUC      : {B.auc.mean() - A.auc.mean():+.3f}")
print(f"\nNote the spread too: the random split reports sd {B.p50.std():.3f} vs {A.p50.std():.3f}")
print("grouped. It looks BOTH better and steadier -- the false confidence is the trap.")


BEFORE / AFTER  -- identical model and features, only the split changes
                              BEFORE: random row split  AFTER: grouped by client
P@50 mean                                        0.962                     0.805
P@50 sd                                          0.029                     0.108
ROC-AUC                                          0.782                     0.710
test base rate                                   0.544                     0.495
clients in BOTH train & test                    31.500                     0.000

INFLATION bought by the dishonest split:
  precision@50 : +0.157
  ROC-AUC      : +0.072

Note the spread too: the random split reports sd 0.029 vs 0.108
grouped. It looks BOTH better and steadier -- the false confidence is the trap.


### What the before/after shows

Same model, same features, same data, and **only the split changed**, and precision@50 fell from ≈0.962 to ≈0.805. That **+0.157 gap is memorisation**: with a random row split, roughly every client has some pages in training, so the model learns each client's typical behaviour and is then tested on more pages from those same clients. It is not learning what a declining page looks like; it is learning what *this client's* pages look like.

The gap is the finding, exactly as the skill describes it. Had I reported the 0.962, every number downstream would have been wrong by an amount nobody could see.

**The more dangerous half is the spread.** The random split reports a standard deviation of ≈0.029 against ≈0.108 for the grouped split. The dishonest design looks *better and steadier at the same time*, producing a tight, confident-looking number. That is what makes it so easy to believe: nothing about it looks broken.

This is also why I asked the paper (Finding 1) whether its holdout was grouped by brand. With 57 brands and pages sharing templates within a brand, the same mechanism is available there, and I measured what it is worth on my own data before raising it.


In [2]:
# REAL FAILURE EXAMPLES under the honest (grouped) split -- the rows that would
# actually waste an editor's week.
g_tr, g_te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
                  .split(df, y, groups=df["client_id"]))
prob = model(NUM).fit(df.iloc[g_tr][NUM + CAT], y[g_tr]).predict_proba(df.iloc[g_te][NUM + CAT])[:, 1]
t = df.iloc[g_te].copy(); t["prob"] = prob; t["label"] = y[g_te]

top50 = t.nlargest(50, "prob")
fp = top50[top50["label"] == 0]
print(f"top-50 queue on held-out clients: {int(top50.label.sum())}/50 correct "
      f"| {len(fp)} would waste an editor's time\n")

show = ["prob", "impressions_90d", "avg_position", "ctr", "clicks_90d",
        "days_since_last_update", "content_type", "trend_direction"]
print("FALSE POSITIVES -- confidently queued, but NOT declining:")
print(fp.nlargest(3, "prob")[show].to_string(index=False))

# The opposite error: genuinely declining pages the model ranked lowest.
missed = t[t["label"] == 1].nsmallest(3, "prob")
print("\nFALSE NEGATIVES -- genuinely declining, ranked near the bottom:")
print(missed[show].to_string(index=False))

print("\nMissed declining pages by impression tier (recall in the top 50):")
recall = (t[t.label == 1].assign(in_top50=lambda d: d.index.isin(top50.index))
            .groupby("impression_tier", observed=True)["in_top50"]
            .agg(declining_pages="size", caught_in_top50="sum"))
print(recall.to_string())
print("\n-> precision@50 says nothing about these. The queue only shows 50 rows;")
print("   every declining page outside it is a miss nobody ever sees.")


top-50 queue on held-out clients: 47/50 correct | 3 would waste an editor's time

FALSE POSITIVES -- confidently queued, but NOT declining:
    prob  impressions_90d  avg_position  ctr  clicks_90d  days_since_last_update    content_type trend_direction
0.857926              468           7.2  0.0           0                     104 keyword article              up
0.829906              206          19.0  0.0           0                      20 keyword article              up
0.829896              612          14.6  0.0           0                      20 keyword article          stable

FALSE NEGATIVES -- genuinely declining, ranked near the bottom:
    prob  impressions_90d  avg_position  ctr  clicks_90d  days_since_last_update    content_type trend_direction
0.031690                1           0.0  0.0           0                      20 keyword article            down
0.053070                2           0.0  0.0           0                      20 keyword article            down
0.05

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# LEAKAGE AUDIT on my final feature set, run against the taxonomy in
# skills/hunting-leakage-and-validating: (1) label-derived, (2) future/overlapping
# windows, (3) decision-derived product flags.

LABEL_DERIVED = {"trend_direction", "trend_pct"}
WINDOW_OVERLAP = {"impressions_last_30d", "impressions_prev_30d",
                  "clicks_last_30d", "clicks_prev_30d",
                  "sessions_last_30d", "sessions_prev_30d"}
IDS = {"content_id", "client_id"}

print("CHECKLIST")
print(f"  1. label-derived in features   : {sorted(set(NUM+CAT) & LABEL_DERIVED) or 'NONE'}")
print(f"  2. overlapping-window features : {sorted(set(NUM+CAT) & WINDOW_OVERLAP) or 'NONE'}")
print(f"  3. product/decision flags      : NONE (the starter export ships none)")
print(f"  4. IDs used as features        : {sorted(set(NUM+CAT) & IDS) or 'NONE'}")
print(f"  5. split grouped by client     : yes (Section 2)")
print(f"  6. base rate printed           : yes, beside every metric")
assert not (set(NUM + CAT) & (LABEL_DERIVED | WINDOW_OVERLAP | IDS)), "LEAK in feature set"

# --- POSITIVE CONTROL --------------------------------------------------------
# A checklist that never fires proves nothing. So deliberately ADD each suspect
# and confirm the score jumps -- if it does not, the HARNESS is broken, not the data.
g_tr, g_te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
                  .split(df, y, groups=df["client_id"]))
print("\n" + "=" * 76)
print("POSITIVE CONTROL -- add the leak on purpose, watch it fire (grouped split)")
print("=" * 76)
control = {}
for name, extra in [("honest (my final feature set)", []),
                    ("+ trend_pct  (label IS this column bucketed)", ["trend_pct"]),
                    ("+ impressions_last_30d & _prev_30d  (label is their ratio)",
                     ["impressions_last_30d", "impressions_prev_30d"])]:
    cols = NUM + extra
    prob = model(cols).fit(df.iloc[g_tr][cols + CAT], y[g_tr]).predict_proba(df.iloc[g_te][cols + CAT])[:, 1]
    auc, p50 = roc_auc_score(y[g_te], prob), precision_at_k(prob, y[g_te])
    control[name] = {"auc": auc, "p50": p50}
    print(f"  {name:<58} AUC {auc:.3f}  P@50 {p50:.3f}")

honest_auc = control["honest (my final feature set)"]["auc"]
leak_auc = control["+ trend_pct  (label IS this column bucketed)"]["auc"]
print(f"\n  The harness DOES detect leakage: AUC {honest_auc:.3f} -> {leak_auc:.3f} when trend_pct enters.")
print("  So the honest number is honest because the features are clean, not because")
print("  the test is blind.")
print(f"\n  The window pair also fires (+{control['+ impressions_last_30d & _prev_30d  (label is their ratio)']['auc'] - honest_auc:.3f} AUC).")
print("  Neither column is 'the label' by name -- but their RATIO is, which is why")
print("  a by-name blocklist is not enough. You have to know how the label was built.")


CHECKLIST
  1. label-derived in features   : NONE
  2. overlapping-window features : NONE
  3. product/decision flags      : NONE (the starter export ships none)
  4. IDs used as features        : NONE
  5. split grouped by client     : yes (Section 2)
  6. base rate printed           : yes, beside every metric

POSITIVE CONTROL -- add the leak on purpose, watch it fire (grouped split)


  honest (my final feature set)                              AUC 0.730  P@50 0.940


  + trend_pct  (label IS this column bucketed)               AUC 1.000  P@50 1.000


  + impressions_last_30d & _prev_30d  (label is their ratio) AUC 0.940  P@50 1.000

  The harness DOES detect leakage: AUC 0.730 -> 1.000 when trend_pct enters.
  So the honest number is honest because the features are clean, not because
  the test is blind.

  The window pair also fires (+0.210 AUC).
  Neither column is 'the label' by name -- but their RATIO is, which is why
  a by-name blocklist is not enough. You have to know how the label was built.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Four sentences I wrote in earlier assignments that go further than my evidence. Each is quoted as written, then rewritten.

---

**1. The one I most want back.** From ML-02 and repeated in ML-03:

> ~~"The transparent rule loses to random. That gap is the project."~~

**Rewritten:** *On the single client-holdout split the shipped pipeline reports, the hand rule scored precision@50 = 0.240 against a 0.391 base rate. I have not tested whether that gap survives repeated splits, so I treat it as one observation, not an established property of the rule.*

Why it needed fixing: I built a whole lane rationale on **one split**. By ML-08 I had measured that single splits move precision@50 by ±0.11 to ±0.18, and in this very notebook split 3 has my forest *losing* to the baseline. A gap of 0.15 on one split is inside that noise. The claim may well be true; I simply never earned it.

---

**2. From ML-08, the interpretation section:**

> ~~"The model is not merely an age detector. That specific worry is now partly retired."~~

**Rewritten:** *Permutation importance ranks `content_age_days` third, and shuffling it costs about 0.021 ROC-AUC, so the model is not relying on age alone. That measures what the model leans on — it does not test whether age confounds staleness, which needs a stratified comparison I have not run.*

Why: I answered a question about **confounding** with evidence about **feature importance**. They are different things. A model can spread a confound across several correlated features and still be driven by it. The age-versus-staleness worry from ML-02 is still open.

---

**3. From ML-07, the weak-picks section:**

> ~~"Precision@50 and business value pull in opposite directions here."~~

**Rewritten:** *Across the five ranking variants I tested on this dataset, the orderings that scored highest at precision@50 concentrated on low-traffic pages. That is an observed pattern in one rule family on one snapshot, not a general property of the lane.*

Why: "pull in opposite directions" states a law. I tested five hand-built orderings on one 90-day export. A different feature or a different K could easily break the pattern.

---

**4. From ML-08, reading the table:**

> ~~"Random forest is the best model."~~

**Rewritten:** *Random forest and gradient boosting are indistinguishable on this evidence — the forest wins 5 of 8 splits by an average of 0.022, well inside the ±0.11 split-to-split spread. I prefer the forest for its lower variance and simpler defaults, which is a decision-support judgement rather than a measured superiority.*

Why: I had the honest analysis in the notebook and still let a superlative into the summary sentence. "Best" implies a ranking the numbers do not support.

---

### The pattern in my own mistakes

All four are the same error: **stating a property of the world when I had measured a property of one run.** They were not careless numbers; every figure was computed correctly. The overreach happened in the sentence *after* the number, where a measurement quietly became a law.

The habit I am taking into ML-10 and the paper: after writing any claim, ask *"how many splits, how many datasets, how many rule variants is this true of?"*, then put that answer in the sentence itself. `observed`, `measured`, and `directional` are not decoration; they are where the sample size goes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere: pseudonymous IDs only
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**What changed in my numbers because of this audit.** Nothing, and that is the point of running it. My Week-5 model was already on a grouped split, so the honest number stands at precision@50 ≈ 0.805 (±0.11). What this notebook adds is the *evidence* that it is honest: the +0.157 I would have gained from a random split, and two positive controls proving the harness detects a leak when one is present.

**What changed in my writing.** Four claims rewritten in Section 4, all of them the same error: stating a property of the world after measuring a property of one run.

